# 第 01 章 · 红杯仿真实验

选择 **CPU**（硬件加速器选「无」），然后点「运行全部」。首次运行会下载实验代码和仿真依赖。最后会显示执行一步动作前后的相机画面，以及机械臂末端的位移。不需要下载演示数据。


In [ ]:
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

work = Path('/content/vla-chapter01')
work.mkdir(exist_ok=True)
repo = work / 'VLA-Step-by-Step'
libero = work / 'LIBERO'
libero_revision = '8f1084e3132a39270c3a13ebe37270a43ece2a01'
source_zip = None  # 未推送时可改成上传到 Colab 的源码 ZIP 路径。

def run(*args, **kwargs):
    subprocess.run([str(arg) for arg in args], check=True, **kwargs)

if source_zip is not None:
    with ZipFile(source_zip) as archive:
        archive.extractall(repo)
elif not repo.exists():
    run('git', 'clone', '--depth', '1', 'https://github.com/Blue0w/VLA-Step-by-Step.git', repo)
lab = repo / 'chapters/01_environment/lab'
requirements = lab / 'requirements.txt'
if not requirements.is_file():
    raise RuntimeError('没有找到实验代码。请从已推送的分支打开，或上传仓库源码 ZIP。')

if not (libero / 'libero/libero/__init__.py').is_file():
    run('git', 'init', libero)
    run('git', '-C', libero, 'fetch', '--depth', '1',
        'https://github.com/Lifelong-Robot-Learning/LIBERO.git', libero_revision)
    run('git', '-C', libero, 'checkout', '--detach', 'FETCH_HEAD')

run('apt-get', '-qq', 'update')
run('apt-get', '-qq', 'install', '-y', 'xvfb', 'xauth', 'libgl1',
    'libglx-mesa0', 'libgl1-mesa-dri')
run(sys.executable, '-m', 'pip', '-q', 'install', 'uv')
uv = [sys.executable, '-m', 'uv']
venv = work / '.venv'
python = venv / 'bin/python'
if not python.exists():
    run(*uv, 'venv', '--python', '3.12', venv)
run(*uv, 'pip', 'install', '--python', python, '-r', requirements, '-e', repo)


In [ ]:
import os

output = work / 'outputs/chapter01'
env = os.environ.copy()
# 将 LIBERO 仓库根目录加入 Python 的模块搜索路径。
env['PYTHONPATH'] = str(libero)
# GLX 在虚拟显示器上使用 CPU 渲染。Agg 只影响仿真子进程。
env.update(MUJOCO_GL='glx', PYOPENGL_PLATFORM='glx',
           LIBGL_ALWAYS_SOFTWARE='1', MPLBACKEND='Agg',
           LIBERO_CONFIG_PATH=str(work / 'libero-config'))
# 首次导入 LIBERO 时选择默认数据目录。
result = subprocess.run(
    ['xvfb-run', '-a', str(python), '-u', '-X', 'faulthandler',
     str(lab / 'demo.py'), '--output-dir', str(output)],
    cwd=repo, env=env, input='n\n', text=True, capture_output=True,
)
print(result.stdout, end='')
print(result.stderr, end='')
if result.returncode != 0:
    raise RuntimeError(f'仿真运行失败（退出码 {result.returncode}）。请查看上方 demo.py 的报错。')


In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(8, 4), layout='constrained')
for axis, label in zip(axes, ('before', 'after')):
    axis.imshow(plt.imread(output / f'{label}.png'))
    axis.set_title(label)
    axis.set_axis_off()
plt.show()
